In [ ]:
!pip install sentence-transformers

In [2]:
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import minsearch

# 1. Cargamos los documentos locales
with open('documents-with-ids.json', 'rt') as f_in:
    documents = json.load(f_in)

# 2. Cargamos nuestro Ground Truth y filtramos SOLO el curso de Machine Learning
df_ground_truth = pd.read_csv('ground-truth-data.csv')
df_ground_truth = df_ground_truth[df_ground_truth.course == 'machine-learning-zoomcamp']
ground_truth = df_ground_truth.to_dict(orient='records')

print(f"Total de preguntas a evaluar (solo ML): {len(ground_truth)}")

# 3. Descargamos el modelo de embeddings para convertir texto a vectores
print("Cargando el modelo de Sentence Transformers...")
model_name = 'multi-qa-MiniLM-L6-cos-v1'
model = SentenceTransformer(model_name)

# 4. Convertimos nuestros documentos a vectores
print("Creando vectores para los documentos (Esto tomará un par de minutos)...")
vectors = []
for doc in tqdm(documents):
    question = doc['question']
    text = doc['text']
    # Unimos la pregunta y la respuesta para tener más contexto
    vector = model.encode(question + ' ' + text)
    vectors.append(vector)

vectors = np.array(vectors)

# 5. Guardamos los vectores en el motor de minsearch
print("Indexando en minsearch VectorSearch...")
vindex = minsearch.vector.VectorSearch(keyword_fields=['course'])
vindex.fit(vectors, documents)

print("¡Motor de búsqueda vectorial listo!")

Total de preguntas a evaluar (solo ML): 1830
Cargando el modelo de Sentence Transformers...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creando vectores para los documentos (Esto tomará un par de minutos)...


  0%|          | 0/948 [00:00<?, ?it/s]

Indexando en minsearch VectorSearch...
¡Motor de búsqueda vectorial listo!


In [3]:
from openai import OpenAI

client = OpenAI()

# 1. Funciones de Búsqueda Vectorial
def minsearch_vector_search(vector, course):
    return vindex.search(
        vector,
        filter_dict={'course': course},
        num_results=5
    )

def question_text_vector(q):
    question = q['question']
    course = q['course']
    
    # Convertimos la pregunta a vector matemático
    v_q = model.encode(question)
    
    return minsearch_vector_search(v_q, course)

# 2. Función para armar el Prompt
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

# 3. Función para consultar al LLM (OpenAI)
def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 4. Función RAG maestra
def rag(query: dict, model='gpt-4o-mini') -> str:
    search_results = question_text_vector(query)
    prompt = build_prompt(query['question'], search_results)
    answer = llm(prompt, model=model)
    return answer

# --- PRUEBA DEL SISTEMA ---
# Tomamos la pregunta número 11 (índice 10) de nuestra lista
pregunta_prueba = ground_truth[10]

print(f"Pregunta original: {pregunta_prueba['question']}")
print("Buscando y generando respuesta...\n")

respuesta_generada = rag(pregunta_prueba)

print(f"Respuesta del LLM:\n{respuesta_generada}")

Pregunta original: Are sessions recorded if I miss one?
Buscando y generando respuesta...

Respuesta del LLM:
Yes, sessions are recorded, so if you miss one, you won't miss any content. You can review the recordings later.
